# 3. Machine Learning Models
Training and evaluating regression models for house price prediction

## Step 0: Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import matplotlib.pyplot as plt

# 1. Примусово прописуємо абсолютний шлях до папки src
# Оскільки ноутбук лежить у notebooks/, '..' піднімає нас у корінь проекту, а далі заходимо в src/
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(project_root, 'src')

if src_path not in sys.path:
    sys.path.append(src_path)

# 2. Стандартні імпорти для дата-сайнсу
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 3. Імпорт вашого локального класу (тепер Python знає, де його шукати)
from train_model import HousePricePredictor

print("✓ Усі модулі та класи успішно завантажено! Можна працювати.")


# import pandas as pd
# import numpy as np
# import sys
# import os
# import matplotlib.pyplot as plt

# # 1. Визначаємо абсолютний шлях до папки src
# project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
# src_path = os.path.join(project_root, 'src')

# # 2. Додаємо цей шлях до системних шляхів Python
# if src_path not in sys.path:
#     sys.path.append(src_path)

# # 3. Імпортуємо потрібні модулі
# from data_preprocessing import DataPreprocessor, load_csv, split_features_target
# from train_model import HousePricePredictor

# print("✓ Paths configured, modules imported successfully!")

# Load and prepare data
df = load_csv(os.path.join(project_root, 'data', 'raw', 'Bengaluru_House_Data.csv'))
preprocessor = DataPreprocessor(df)
df_clean = preprocessor.load_and_clean()
X, y = split_features_target(df_clean, target_col='price')

print(f"✓ Data prepared: {X.shape[0]} samples, {X.shape[1]} features")

INFO:data_preprocessing:Original dataset shape: (13320, 9)
INFO:data_preprocessing:After removing duplicates: (12791, 9)
d:\learning\hsh2\final_project\DataScienceProjekt\src\data_preprocessing.py:45: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  self.df[col].fillna(self.df[col].median(), inplace=True)
d:\learning\hsh2\final_project\

✓ Paths configured, modules imported successfully!
✓ Data prepared: 12791 samples, 8 features


## Step 1: Initialize and Train Models

In [ ]:
# Train models
print("Training models...")
predictor = HousePricePredictor()
predictor.preprocess_data(X, y, test_size=0.2)
models = predictor.train_models()

print(f"\n✓ Trained {len(models)} models:")
for model_name in models.keys():
    print(f"  - {model_name}")

## Step 2: Model Evaluation

In [ ]:
# Evaluate models
print("\nEvaluating models...")
results = predictor.evaluate_models()

# Display results
print("\n" + "=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)

results_df = pd.DataFrame(results).T
results_df = results_df[['R2', 'RMSE', 'MAE']]
print(results_df.to_string())

best_model_name, best_model = predictor.get_best_model()
print(f"\n✓ Best Model: {best_model_name}")
print(f"  R² Score: {results[best_model_name]['R2']:.4f}")

## Step 3: Prediction Example

In [ ]:
# Make predictions on test set
y_pred = predictor.predict(predictor.X_test, model_name=best_model_name)

# Show some examples
print("\nSample Predictions vs Actual:")
print("=" * 50)
comparison = pd.DataFrame({
    'Actual': predictor.y_test.values[:10],
    'Predicted': y_pred[:10],
    'Error': abs(predictor.y_test.values[:10] - y_pred[:10])
})
print(comparison.to_string())
print(f"\nAverage Absolute Error: {comparison['Error'].mean():.2f}")

## Step 4: Visualization

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Actual vs Predicted
axes[0].scatter(predictor.y_test, y_pred, alpha=0.5)
axes[0].plot([predictor.y_test.min(), predictor.y_test.max()], 
             [predictor.y_test.min(), predictor.y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title(f'{best_model_name} - Actual vs Predicted')
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals
residuals = predictor.y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Model training and evaluation completed!")